In [66]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize

In [4]:
nltk.download('punkt_tab')
nltk.download('stopwords')

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from nltk.stem import PorterStemmer


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\kinet\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kinet\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# making 3D plots

In [4]:
import plotly.express as px

In [5]:
dataframe = pd.read_json(r"D:/YouTube-Likes-predictor/kzg.json")
 # Normalize the 'videos' field in the JSON
dataframe = pd.json_normalize(dataframe['videos'])
dataframe['likes'] = dataframe['likes'].apply(convert_likes)
dataframe['views'] = dataframe['views'].str.replace(',', '').astype(int)

In [6]:
dataframe.head()

,video title,views,days ago,products,likes
0,What Actual Aliens Might Look Like,4853082,19,2,191000.0
1,The Real Reason Why You Have Allergies,4497271,73,12,163000.0
2,Is The World Getting More Violent?,4042650,75,10,187000.0
3,Let's Talk About The Last 10 Months,1389757,80,9,58000.0
4,Black Hole's Evil Twin - Gravastars Explained,4712586,82,9,190000.0


In [12]:
fig = px.scatter(dataframe, 
                 x='days ago', 
                 y='views', 
                #  text='video title',
                 size='likes',  # Points sized by this column
                 color='products',  # Different colors for categories
                 title='YouTube Video Statistics',
                 labels={'days ago': 'Days Since Upload', 'views': 'View Count'})

In [13]:
fig.update_traces(
    marker=dict(line=dict(width=1, color='DarkSlateGrey')),
    hovertemplate=
    '<b>Title</b>: %{customdata[0]}<br>' +  # Access video title from customdata
    '<b>Views</b>: %{y:,}<br>' +
    '<b>Days Ago</b>: %{x}<br>' +
    '<b>Likes</b>: %{marker.size:,}<br>' +
    '<b>Products</b>: %{customdata[1]}<br>',
    customdata=dataframe[['video title', 'products']]  # Pass both title and products as customdata
)

In [14]:
fig.update_layout(
    showlegend=True,
    hovermode='closest',
    plot_bgcolor='white',
    paper_bgcolor='white',
    title_x=0.5,
    xaxis_title="Days Since Upload",
    yaxis_title="View Count",
    yaxis=dict(tickformat=",")
)


# Bad word analysis

In [69]:
with open(r"D:\YouTube-Likes-predictor\bad words don't open.txt", 'r') as f:
    bad_words = set(f.read().splitlines())

In [73]:
def detect_bad_words(text : str, bad_words : list):
    tokens = word_tokenize(text.lower())  # Tokenize and convert to lowercase
    # Return only the words that match the bad_words set
    return [word for word in tokens if word in bad_words]

In [75]:
dataframe['bad_words'] = dataframe['video title'].apply(lambda x: detect_bad_words(x, bad_words))
dataframe['bad_word_count'] = dataframe['bad_words'].apply(len)